# Brain Tumor Segmentation and Multi-Task Classification Pipeline

This notebook demonstrates the end-to-end medical image analysis workflow originally implemented in the Colab notebook, now refactored into a clean, modular repository structure. 

### Pipeline Stages:
1. **3D Data Loading & Transforms**: Load multi-modal NIfTI scans (T1, T1c, T2, FLAIR) and apply MONAI spatial normalization, foreground cropping, RAS orientation, and volume resizing.
2. **SwinUNETR Segmentation**: Load fine-tuned SwinUNETR weights to generate 3D probability maps for Enhancing Tumor (ET), Tumor Core (TC), and Whole Tumor (WT).
3. **Feature Extraction & Precomputation**:
   * Extract Stage 4 bottleneck features from the SwinUNETR encoder ($768$-dimensional vector).
   * Compute 3D morphological descriptors (volume, surface area, compactness, sphericity) from the segmentations.
   * Extract Test-Time Augmentation (TTA) model confidence maps.
4. **Confidence-Gated Fusion**: Modulate representations based on segmentation confidence maps and spatial morphology gates.
5. **Multi-Task Classification**: Classify **IDH Status**, **WHO CNS Grade**, and **MGMT Promoter Methylation** using the fused representations.

In [ ]:
import os
import sys
import torch

# Ensure project root is in the system path for imports
project_root = os.path.dirname(os.getcwd())
if project_root not in sys.path:
    sys.path.append(project_root)

print(f"Project root added to path: {project_root}")

## 1. Load Configurations

In [ ]:
from configs.config import cfg

print("=== Config Parameters ===")
print(f"Device             : {cfg.DEVICE}")
print(f"Spatial size       : {cfg.SPATIAL_SIZE}")
print(f"Gating ablation    : {cfg.GATE_MODE}")
print(f"Classifier LR      : {cfg.LR}")
print(f"Dataset Path       : {cfg.UCSF_DIR}")
print(f"Metadata CSV Path  : {cfg.UCSF_CSV}")

## 2. Load Patient Dictionaries and Preprocessing Transforms

In [ ]:
from src.data.dataset import build_ucsf_dicts, stratified_split
from src.data.transforms import get_ucsf_transforms

# Build metadata list of patients with valid paths
try:
    all_dicts = build_ucsf_dicts(cfg.UCSF_CSV, cfg.UCSF_DIR)
    train_d, val_d, test_d = stratified_split(all_dicts)
    print(f"Total Patients loaded: {len(all_dicts)}")
    print(f"Split - Train: {len(train_d)} | Val: {len(val_d)} | Test: {len(test_d)}")
except FileNotFoundError as e:
    print(f"Dataset not found at pathways: {e}")
    print("To run mock/dry runs, update paths in configs/config.py or download the UCSF-PDGM-v5 dataset.")

## 3. SwinUNETR Segmentation and Feature Extraction

In [ ]:
from src.models.backbone import build_swinunetr, extract_encoder_features, tta_predict
from src.utils.helpers import extract_morphological_features, morphology_to_tensor

# Instantiate the SwinUNETR model (it automatically frozen if config.FREEZE_BACKBONE is True)
model = build_swinunetr(weights_path=cfg.WEIGHTS)

print("Backbone model prepared.")

## 4. Confidence-Gated Fusion & Multi-Task Classifier

In [ ]:
from src.models.classifier import BrainTumorClassifier

# Initialize classifier head
classifier = BrainTumorClassifier(
    encoder_dim=cfg.ENCODER_DIM,
    morph_dim=cfg.MORPH_FEATURES,
    fused_dim=cfg.FUSED_DIM,
    gate_mode=cfg.GATE_MODE
).to(cfg.DEVICE)

print(classifier)
print(f"Classifier parameters: {sum(p.numel() for p in classifier.parameters())/1e6:.2f}M")

## 5. Training and Evaluation Scripts

To run the training pipelines, feature cache precomputations, ablation studies, and McNemar statistical tests, you can use the command-line entrypoint scripts or call the evaluators programmatically.

### Example: Running evaluation using precomputed feature cache loaders

In [ ]:
from torch.utils.data import DataLoader
from src.data.dataset import CachedDataset
from src.evaluation.evaluator import evaluate_classifier

if os.path.exists(cfg.SLIM_CACHE) and len(os.listdir(cfg.SLIM_CACHE)) > 0:
    test_files = [os.path.join(cfg.SLIM_CACHE, f"{d['patient_id']}.pt") for d in test_d]
    test_loader = DataLoader(CachedDataset(test_files), batch_size=1, shuffle=False)
    
    # Load best checkpoint weights
    best_clf_ckpt = os.path.join(cfg.CKPT_DIR, 'clf_best_seed0.pth')
    if os.path.exists(best_clf_ckpt):
        ckpt = torch.load(best_clf_ckpt, map_location=cfg.DEVICE)
        classifier.load_state_dict(ckpt['model_state'])
        print("Loaded classifier weights.")
        
        test_metrics = evaluate_classifier(classifier, test_loader, device=cfg.DEVICE)
        for task, met in test_metrics.items():
            print(f"\n--- {met['name']} Test Metrics ---")
            print(f"  Accuracy  : {met['accuracy']:.3f}")
            print(f"  F1-Score  : {met['f1']:.3f}")
            print(f"  AUC Score : {met['auc']:.3f}")
            print(f"  Sens/Spec : {met['sensitivity']:.3f} / {met['specificity']:.3f}")
    else:
        print(f"Classifier checkpoint not found at {best_clf_ckpt}. Train the classifier head first.")
else:
    print("Precomputed slim cache directory not found or empty. Please run precomputation caching first.")